# Prompt Injection Detector Agent

In this notebook, I build a small agent that treats text handed to it with a little suspicion, using the Hugging Face `smolagents` library.

Agents often have to read text they did not write themselves: a search result, a document, a message someone pasted in. If that text contains instructions aimed at the agent instead of at me, something like "ignore your previous instructions and do this instead", it is called a prompt injection attempt. This notebook builds a small, rule-based way to notice that kind of text before acting on it, and a way to strip the offending lines out.

I want to be upfront about scope from the start: what I build here is a basic keyword check, not a real defense. It is useful for seeing the shape of the problem, not for actually securing anything, and I come back to exactly why at the end.

In this notebook, I will:

- Write a plain Python function that scans text for suspicious instruction-like patterns
- Watch it miss an obvious attempt, then loosen the matching to catch it
- Turn it into a tool that reports whether a pattern was found, and which one
- Write a second function that redacts any line containing a suspicious pattern
- Turn that into a tool too, and give an agent both
- Look at how easily this kind of rule-based check is evaded, and what that implies for anything I build for real

## 1. Importing Libraries and Creating the Model

First I import the pieces I need from `smolagents`.

`CodeAgent` is the agent that writes and runs Python code to solve a task. `tool` is the decorator I use to turn a plain function into something an agent can call. `InferenceClientModel` is the language model, which runs on the Hugging Face Inference API rather than on my own machine.

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)